# Data Generation (H10) for ML


In [1]:
from pathlib import Path
from glob import glob
import pickle

import pandas as pd

## 1. Configuration and Kubios column map

`KUBIOS_COL_MAP` selects and renames the Kubios CSV HRV columns, and the
`DATA_ROOT` and `PRE_WINDOW_MIN` variables centralize the data root and
windowing parameters.


In [2]:
# Maps the 0-based column position in the Kubios CSV -> the HRV name to keep.
KUBIOS_COL_MAP: dict[int, str] = {
    1: "timestamp",
    7: "pns_index",
    8: "sns_index",
    9: "stress_index",
    10: "ee_activity",
    12: "trimp_intensity",
    14: "mean_RR",
    15: "sdnn",
    16: "mean_HR",
    20: "rmssd",
    21: "NN",
    22: "pNN",
    25: "DC",
    27: "AC",
    38: "VLF",
    39: "LF",
    40: "HF",
    43: "lf_hf_ratio",
    45: "SD1",
    46: "SD2",
    47: "SD2SD1_ratio",
    48: "ApEn",
    49: "SampEn",
}



DATA_ROOT = Path("../R21_data")
PRE_WINDOW_MIN = 15


## 2. Kubios CSV parsing

Read the raw Kubios 60s/10s CSV exports, keep the mapped columns, and produce a
per-participant feature pickle with an integer-second timestamp.

In [3]:
def _create_kubios_df(raw: pd.DataFrame) -> pd.DataFrame:
    """Select and clean the mapped Kubios columns from a raw export.

    Args:
        raw: Raw dataframe read from a single Kubios CSV (after header skip).

    Returns:
        A dataframe limited to the mapped, numeric Kubios columns with parsed
        timestamps and trailing non-time rows removed.
    """
    selected = raw.iloc[:, list(KUBIOS_COL_MAP.keys())].copy()  # keep only the columns in the map, by position
    selected.columns = [KUBIOS_COL_MAP[idx] for idx in KUBIOS_COL_MAP]  # rename them to the HRV names

    selected["timestamp"] = selected["timestamp"].astype(str)
    cut_idx = None
    for i, value in enumerate(selected["timestamp"].tolist()):
        if ":" not in value:  # first row without a clock time marks the start of the CSV's summary footer
            cut_idx = i
            break
    if cut_idx is not None:
        selected = selected.iloc[:cut_idx]  # drop the footer rows

    selected["timestamp"] = pd.to_datetime(selected["timestamp"], errors="coerce")
    selected = selected.dropna(subset=["timestamp"]).reset_index(drop=True)  

    numeric_cols = [c for c in selected.columns if c != "timestamp"]
    selected[numeric_cols] = selected[numeric_cols].apply(pd.to_numeric, errors="coerce")  # force feature columns to numeric
    return selected


def kubios_csv_to_features(participant_id: str) -> pd.DataFrame:
    """Parse Kubios CSV outputs and save participant-level Kubios features.

    Args:
        participant_id: Participant identifier.

    Returns:
        The merged, time-sorted Kubios feature dataframe (also written to disk).
    """
    csv_root = DATA_ROOT / "kubios_csvs" / participant_id / "kubios_result_60s_10s"
    csv_files = sorted(glob(str(csv_root / "*.csv")))

    kubios_dfs: list[pd.DataFrame] = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, skiprows=157)  # Kubios metadata header is 157 lines long
        out = _create_kubios_df(df)
        out["source_file"] = Path(csv_file).name  # remember which export each row came from
        kubios_dfs.append(out)

    if not kubios_dfs:  # participant had no Kubios CSVs
        return pd.DataFrame(columns=list(KUBIOS_COL_MAP.values()) + ["source_file"])

    merged = (
        pd.concat(kubios_dfs, ignore_index=True)
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    merged["ts_int"] = (merged["timestamp"] - pd.Timestamp("1970-01-01")).dt.total_seconds()  # epoch seconds, used to join against other signals

    out_dir = DATA_ROOT / "kubios_features" / participant_id
    out_dir.mkdir(parents=True, exist_ok=True)
    merged.to_pickle(out_dir / f"{participant_id}_kubios_60s10s.pkl")
    return merged

## 3. Pre-15 feature labeling

Label each Kubios row as smoking or non-smoking based on the saved window timestamp
lists, then build the combined per-participant feature set.


In [4]:
def _load_window_lists(
    participant_id: str,
) -> tuple[list[list[float]], list[list[float]]]:
    """Load the saved smoking and non-smoking window timestamp lists.

    Args:
        participant_id: Participant identifier.

    Returns:
        A tuple ``(smoke_windows, nonsmoke_windows)`` of timestamp lists.
    """
    window_dir = DATA_ROOT / "window_ts" / participant_id
    smoke_path = (
        window_dir
        / f"{participant_id}_pre{PRE_WINDOW_MIN}smoke_alldf_tsint_window_{PRE_WINDOW_MIN}"
    )
    nonsmoke_path = (
        window_dir
        / f"{participant_id}_pre{PRE_WINDOW_MIN}_nonsmoke_alldf_tsint_window_{PRE_WINDOW_MIN}"
    )

    with smoke_path.open("rb") as handle:
        smoke_windows = pickle.load(handle)  # each entry is a list of timestamps for one smoking event
    with nonsmoke_path.open("rb") as handle:
        nonsmoke_windows = pickle.load(handle)  # same, for control (non-smoking) windows
    return smoke_windows, nonsmoke_windows


def _window_label_rows(
    kubios_df: pd.DataFrame,
    windows: list[list[float]],
    label: int,
    start_index: int,
) -> pd.DataFrame:
    """Select Kubios rows falling inside each window and tag them with a label.

    Args:
        kubios_df: Participant Kubios feature dataframe with a ``ts_int`` column.
        windows: List of timestamp lists, one per event window.
        label: Binary smoking label to assign (1 = smoking, 0 = non-smoking).
        start_index: Offset for ``smoking_index`` so events stay unique.

    Returns:
        A concatenated dataframe of labeled rows (empty if nothing matched).
    """
    rows: list[pd.DataFrame] = []
    for idx, ts_window in enumerate(windows):
        ts_min, ts_max = min(ts_window), max(ts_window)  # window spans from its earliest to latest timestamp
        frame = kubios_df[
            (kubios_df["ts_int"] >= ts_min) & (kubios_df["ts_int"] <= ts_max)
        ].copy()  # Kubios rows that fall inside this window
        if frame.empty:
            continue  # no Kubios data overlapped this window
        frame["smoking_label"] = label
        frame["smoking_index"] = start_index + idx  # unique id per event (offset keeps smoke/non-smoke distinct)
        rows.append(frame)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=kubios_df.columns)


def generate_h10_features_pre15(participant_id: str) -> pd.DataFrame:
    """Create the participant-level pre15 feature dataframe.

    Args:
        participant_id: Participant identifier.

    Returns:
        The combined smoking/non-smoking feature dataframe (also written to disk).
    """
    kubios_path = (
        DATA_ROOT / "kubios_features" / participant_id / f"{participant_id}_kubios_60s10s.pkl"
    )
    kubios_df = pd.read_pickle(kubios_path)

    smoke_windows, nonsmoke_windows = _load_window_lists(participant_id)

    smoke_df = _window_label_rows(kubios_df, smoke_windows, label=1, start_index=0)
    nonsmoke_df = _window_label_rows(
        kubios_df, nonsmoke_windows, label=0, start_index=len(smoke_windows)
    )  # start after the smoke ids so event indices never collide

    merged = pd.concat([smoke_df, nonsmoke_df], ignore_index=True)
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged["p_id"] = participant_id  # tag rows with the participant for later grouped CV

    out_dir = DATA_ROOT / "features_around_events" / participant_id / "pre15min"
    out_dir.mkdir(parents=True, exist_ok=True)
    merged.to_pickle(out_dir / f"{participant_id}_h10_features_pre15.pkl")
    return merged

## 4. Combine and orchestrate

`combine_pre15_features_for_participants` merges per-participant pickles into one
training table, and `process_participant` runs the full per-participant flow.


In [5]:
def combine_pre15_features_for_participants(
    participant_ids: list[str],
) -> pd.DataFrame:
    """Combine participant feature pickles into one dataset.

    Args:
        participant_ids: Iterable of participant identifiers to combine.

    Returns:
        The combined feature dataframe across all participants (also written to disk).
    """
    all_frames: list[pd.DataFrame] = []
    for participant_id in participant_ids:
        path = (
            DATA_ROOT
            / "features_around_events"
            / participant_id
            / "pre15min"
            / f"{participant_id}_h10_features_pre15.pkl"
        )
        if path.exists():  # skip participants that haven't been processed yet
            all_frames.append(pd.read_pickle(path))

    combined = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    out_dir = DATA_ROOT / "features_around_events" / "H10_features"
    out_dir.mkdir(parents=True, exist_ok=True)
    combined.to_pickle(out_dir / f"h10_features_everyone_pre{PRE_WINDOW_MIN}_clean.pkl")  # the table the BiLSTM notebook loads
    return combined


def process_participant(participant_id: str) -> pd.DataFrame:
    """Run the full per-participant ML data generation flow.

    Args:
        participant_id: Participant identifier.

    Returns:
        The participant pre15 feature dataframe.
    """
    kubios_csv_to_features(participant_id)  # 1. parse raw Kubios CSVs -> feature pickle
    return generate_h10_features_pre15(participant_id)  # 2. label windows and build the feature table

## 5. Run

Set the participant id and generate its feature set.


In [6]:
process_participant("sample01")  # only sample01 has raw Kubios CSVs to parse

# Combine all available per-participant tables into the BiLSTM training dataset.
participant_ids = ["sample01", "sample02", "sample03", "sample04", "sample05"]
features_df = combine_pre15_features_for_participants(participant_ids)
features_df["p_id"].value_counts()


p_id
sample01    360
sample02    360
sample03    360
sample04    360
sample05    360
Name: count, dtype: int64